In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# 数据示例
torch.manual_seed(0)
x_train = torch.randn(10, 4, 3)  # 10个样本，每个样本有4个时间步，每个时间步3个特征
y_train = torch.randn(10, 1)     # 10个目标值，回归任务

In [6]:
# 自定义GRU模型
class CustomGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(CustomGRU, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.W_z = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_r = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_h = nn.Linear(input_size + hidden_size, hidden_size)
        
        self.fc = nn.Linear(hidden_size, 1)  # 用于回归输出
        
    def forward(self, x, hidden):
        for t in range(x.size(1)):
            combined = torch.cat((x[:, t, :], hidden), dim=1)
            z_t = torch.sigmoid(self.W_z(combined))
            r_t = torch.sigmoid(self.W_r(combined))
            combined_candidate = torch.cat((x[:, t, :], hidden * r_t), dim=1)
            h_tilde = torch.tanh(self.W_h(combined_candidate))
            hidden = (1 - z_t) * hidden + z_t * h_tilde
            
        output = self.fc(hidden)
        return output, hidden

In [7]:
# 超参数
input_size = 3
hidden_size = 5
num_epochs = 100
learning_rate = 0.01

# 模型、损失函数和优化器
model = CustomGRU(input_size, hidden_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 训练
for epoch in range(num_epochs):
    hidden = torch.zeros(x_train.size(0), hidden_size)  # 初始化隐藏状态
    outputs, hidden = model(x_train, hidden)
    
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 测试输出
print("\nFinal output after training:", outputs)


Epoch [10/100], Loss: 0.8648
Epoch [20/100], Loss: 0.6612
Epoch [30/100], Loss: 0.4480
Epoch [40/100], Loss: 0.2816
Epoch [50/100], Loss: 0.1896
Epoch [60/100], Loss: 0.1025
Epoch [70/100], Loss: 0.0375
Epoch [80/100], Loss: 0.0083
Epoch [90/100], Loss: 0.0018
Epoch [100/100], Loss: 0.0009

Final output after training: tensor([[ 1.3225],
        [ 1.2073],
        [ 0.4384],
        [-1.7254],
        [-1.3476],
        [ 0.9076],
        [ 0.7389],
        [ 0.0874],
        [ 0.2295],
        [ 0.5613]], grad_fn=<AddmmBackward0>)


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# 模拟数据
torch.manual_seed(0)
x_train = torch.randn(10, 4, 3)  # 10个样本，每个样本有4个时间步，每个时间步3个特征
y_train = torch.randn(10, 1)     # 10个目标值

# 定义一个简单的GRU模型
class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleGRU, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)  # 用最后时间步的隐藏状态预测输出
    
    def forward(self, x):
        # 初始化隐藏状态
        h0 = torch.zeros(1, x.size(0), hidden_size)  # num_layers=1, (1, batch_size, hidden_size)
        
        # GRU前向传播
        out, hn = self.gru(x, h0)  # out包含所有时间步的输出, hn是最后时间步的隐藏状态
        
        # 使用最后时间步的隐藏状态进行预测
        out = self.fc(hn[-1])  # hn[-1] 取最后一层的隐藏状态
        return out

# 超参数
input_size = 3      # 输入特征维度
hidden_size = 5     # GRU隐藏层大小
output_size = 1     # 输出维度
num_epochs = 100    # 训练轮数
learning_rate = 0.01

# 实例化模型、损失函数和优化器
model = SimpleGRU(input_size, hidden_size, output_size)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 训练模型
for epoch in range(num_epochs):
    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 测试模型
print("\nFinal output after training:", outputs)


Epoch [10/100], Loss: 0.8473
Epoch [20/100], Loss: 0.6851
Epoch [30/100], Loss: 0.4799
Epoch [40/100], Loss: 0.2501
Epoch [50/100], Loss: 0.0822
Epoch [60/100], Loss: 0.0086
Epoch [70/100], Loss: 0.0016
Epoch [80/100], Loss: 0.0013
Epoch [90/100], Loss: 0.0005
Epoch [100/100], Loss: 0.0001

Final output after training: tensor([[ 1.3921],
        [ 1.1847],
        [ 0.4430],
        [-1.7323],
        [-1.3428],
        [ 0.8784],
        [ 0.7718],
        [ 0.0559],
        [ 0.2390],
        [ 0.5485]], grad_fn=<AddmmBackward0>)
